# CWA IFT vs Unrolled K=50 vs GELU Peak GPU Memory Benchmark

This notebook benchmarks **peak GPU memory** for three forward+backward computation modes of the **Curie-Weiss Activation (CWA)** at layer widths n∈{256, 1024, 4096}, batch=64, K_max=50, J_raw=4.0 (J≈0.982), across near-critical (x_scale=0.1) and saturated (x_scale=1.0) regimes.

**Three methods compared:**
1. **CWA-IFT**: Custom `torch.autograd.Function` using implicit function theorem — stores only O(n) activations regardless of iteration depth K.
2. **CWA-Unrolled**: Runs all K=50 iterations through the autograd tape — O(K×n) memory.
3. **GELU Baseline**: `nn.Linear(n,n) + nn.GELU()` — O(n²) weight matrix dominates at large n.

**Key finding**: IFT achieves O(n) memory overhead, using 69–84% less memory than unrolled K=50 at large widths, and 90% less than GELU at n=4096.

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# loguru is NOT pre-installed on Colab
_pip('loguru==0.7.3')

# Core packages (pre-installed on Colab, install locally only)
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'matplotlib==3.10.0', 'torch==2.9.0')

In [ ]:
import gc
import json
import sys
from pathlib import Path
from typing import Callable

import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from loguru import logger

logger.remove()
logger.add(sys.stdout, level="INFO", format="{time:HH:mm:ss}|{level:<7}|{message}")

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-2f6f35-curie-weiss-activation-formal-verificati/main/round-4/experiment-1/demo/mini_demo_data.json"

import json, os

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception: pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f: return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
data = load_data()
examples = data["datasets"][0]["examples"]
meta = data["metadata"]
print(f"Loaded {len(examples)} examples")
print(f"Widths tested: {meta['widths_tested']}")
print(f"x_scales: {meta['x_scales_tested']}")
print(f"Device used for original run: {meta['device']}")
print(f"K_max={meta['K_max']}, batch={meta['batch_size']}, J_sigmoid={meta['J_sigmoid']:.4f}")

## Configuration

All tunable parameters are defined here. The defaults below are **minimum values** for a quick demo. 
Commented-out lines show the original values used in the paper's full benchmark run.

**Note**: The memory measurement cells require a CUDA GPU. If none is available, those cells are skipped and pre-computed results from the loaded data are shown instead.

In [ ]:
# ── Tunable parameters ───────────────────────────────────────────────────────
# Minimum values for quick demo; original paper values in comments

WIDTHS   = [256]           # original: [256, 1024, 4096]
X_SCALES = [0.1]           # original: [0.1, 1.0]
BATCH    = 4               # original: 64
K_MAX    = 5               # original: 50
J_RAW_FIXED = 4.0          # sigmoid(4.0) ≈ 0.982 — fixed across all runs
N_WARMUP = 1               # original: 3
N_MEASURE = 2              # original: 5
TOL      = 1e-4            # original: 1e-6

# ── Hardware detection ───────────────────────────────────────────────────────
HAS_GPU = torch.cuda.is_available()
DEVICE  = torch.device("cuda" if HAS_GPU else "cpu")

print(f"Device: {DEVICE}")
if HAS_GPU:
    props = torch.cuda.get_device_properties(0)
    VRAM_GB = props.total_memory / 1e9
    print(f"GPU: {props.name}, VRAM: {VRAM_GB:.1f} GB")
    _free, _total = torch.cuda.mem_get_info(0)
    torch.cuda.set_per_process_memory_fraction(min(14 * 1024**3 / _total, 0.90))
else:
    print("No GPU found — benchmark cells will be skipped; pre-computed results will be shown.")

J_sigmoid = float(torch.sigmoid(torch.tensor(J_RAW_FIXED)).item())
print(f"J_raw={J_RAW_FIXED}, J=sigmoid(J_raw)={J_sigmoid:.4f}")

## CWA-IFT: Implicit Function Theorem Backward

The CWA fixed point satisfies `m* = (1/n) Σ tanh(x_k + J·m*)`.  
Instead of unrolling iterations through the autograd tape, we:
1. Run fixed-point iteration under `torch.no_grad()` → **zero intermediate tensors stored**
2. Apply the closed-form IFT gradient:
   `∂L/∂x_k = s_k · (g_k + J · Σ(g_i·s_i) / (n·(1 - J·s̄)))`

This gives **O(n) memory** with respect to iteration depth K.

In [ ]:
class CWAIFTFunction(torch.autograd.Function):
    """CWA forward via fixed-point iteration (no intermediate activations stored),
    backward via closed-form IFT gradient (O(n) memory, O(1) w.r.t. K).

    From DEQ theory (arXiv:1909.01377) + CWA scalar fixed-point collapse:
      ∂L/∂x_k = s_k * (g_k + J * Σ_i(g_i*s_i) / (n*(1-J*s̄)))
      ∂L/∂J   = Σ_{b,k}(g_{b,k} * s_{b,k}) * m*_b / (1 - J*s̄_b)
    where s_k = sech²(x_k + J*m*) = 1 - tanh²(x_k + J*m*)
    and   s̄ = mean(s_k over k)
    """

    @staticmethod
    def forward(ctx, x: torch.Tensor, J: torch.Tensor, K_max: int, tol: float):
        # x: (B, n), J: scalar tensor
        B, n = x.shape
        with torch.no_grad():
            m = torch.zeros(B, 1, device=x.device, dtype=x.dtype)
            k_actual = 0
            for k in range(K_max):
                m_new = torch.mean(torch.tanh(x + J * m), dim=1, keepdim=True)
                delta = torch.max(torch.abs(m_new - m)).item()
                m = m_new
                k_actual = k + 1
                if delta < tol:
                    break

        # Re-compute y using saved m* (no grad needed here — backward handles it)
        with torch.no_grad():
            y = torch.tanh(x + J * m)          # (B, n)
            s_bar = torch.mean(1.0 - y ** 2, dim=1, keepdim=True)  # (B, 1)

        # Save only what backward needs: no intermediate activations
        ctx.save_for_backward(x, J, m, y, s_bar)
        ctx.B = B
        ctx.n = n
        ctx.k_actual = k_actual
        return y

    @staticmethod
    def backward(ctx, grad_output: torch.Tensor):
        x, J, m, y, s_bar = ctx.saved_tensors
        B, n = ctx.B, ctx.n

        # s_{b,k} = sech²(x_{b,k} + J*m*_b) = 1 - y_{b,k}²
        s = 1.0 - y ** 2                                                    # (B, n)

        # IFT: denominator clamp prevents divide-by-zero near criticality
        one_minus_Jsbar = (1.0 - J * s_bar).clamp(min=1e-3)                # (B, 1)

        # Σ_k g_k * s_k  for each batch element
        sum_gs = torch.sum(grad_output * s, dim=1, keepdim=True)             # (B, 1)

        # ∂L/∂x_{b,k} = s_{b,k} * (g_{b,k} + J * sum_gs_b / (n * (1-J*s̄_b)))
        grad_x = s * (grad_output + J * sum_gs / (n * one_minus_Jsbar))     # (B, n)

        # ∂L/∂J = Σ_b [ sum_gs_b * m*_b / (1 - J*s̄_b) ]
        grad_J = torch.sum(sum_gs * m / one_minus_Jsbar)                     # scalar

        return grad_x, grad_J, None, None


def cwa_ift_forward(x: torch.Tensor, J: torch.Tensor, K_max: int = 50, tol: float = 1e-6) -> torch.Tensor:
    return CWAIFTFunction.apply(x, J, K_max, tol)

## CWA-Unrolled: Autograd Tape Through All K Iterations

Each iteration `m = mean(tanh(x + J·m))` adds a `(B,1)` node to the autograd graph.  
With K=50, this stores **50 intermediate tensors** — O(K·n) total memory overhead.

In [ ]:
def cwa_unrolled_forward(x: torch.Tensor, J: torch.Tensor, K_max: int = 50) -> torch.Tensor:
    """Stores K intermediate (B,1) tanh results on the autograd tape."""
    B, n = x.shape
    m = torch.zeros(B, 1, device=x.device, dtype=x.dtype)
    for _ in range(K_max):
        # Each iteration adds a (B,1) node to the graph; total K * (B,1) nodes
        m = torch.mean(torch.tanh(x + J * m), dim=1, keepdim=True)
    y = torch.tanh(x + J * m)
    return y

## GELU Baseline

`nn.Linear(n,n) + nn.GELU()` stores input activations `(B,n)` and weight gradients `(n,n)`.  
At large n, the n×n weight matrix dominates memory — O(n²) scaling.

In [ ]:
class GELUBaseline(nn.Module):
    def __init__(self, n: int):
        super().__init__()
        self.linear = nn.Linear(n, n, bias=False)
        self.act = nn.GELU()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.act(self.linear(x))

## Memory Measurement Harness

For each configuration:
1. Run `n_warmup` warm-up passes (forward+backward) to stabilize CUDA memory allocations
2. Run `n_measure` measurement passes, recording `torch.cuda.max_memory_allocated()` after each
3. Report mean ± std of peak allocations in MB

**Regime diagnostic**: Compute J·s̄ (susceptibility proxy) at the fixed point to verify near-critical vs saturated regime.

In [ ]:
def _zero_grads(*tensors):
    for t in tensors:
        if isinstance(t, torch.Tensor) and t.grad is not None:
            t.grad = None


def measure_peak_memory_mb(
    fn: Callable,
    n_warmup: int = 3,
    n_measure: int = 5,
) -> tuple[float, float]:
    """Returns (mean_peak_MB, std_peak_MB) over n_measure runs.

    Each run: reset_peak → fn() → sum().backward() → record peak.
    Uses min over runs to reduce noise from other processes.
    """
    if not HAS_GPU:
        raise RuntimeError("GPU required for CUDA memory measurements")

    # Warmup
    for _ in range(n_warmup):
        torch.cuda.synchronize()
        result = fn()
        loss = result.sum()
        loss.backward()
        torch.cuda.synchronize()
        del result, loss
        gc.collect()
        torch.cuda.empty_cache()

    peaks = []
    for _ in range(n_measure):
        gc.collect()
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()

        result = fn()
        loss = result.sum()
        loss.backward()
        torch.cuda.synchronize()

        peak_mb = torch.cuda.max_memory_allocated() / 1e6
        peaks.append(peak_mb)

        del result, loss
        gc.collect()
        torch.cuda.empty_cache()

    return float(np.mean(peaks)), float(np.std(peaks))


def compute_jsbar(x_data: torch.Tensor, J: float, K_max: int = 200) -> tuple[float, float]:
    """Compute J*s̄ and s̄ at fixed point m* via extended iteration."""
    with torch.no_grad():
        m = torch.zeros(x_data.shape[0], 1, device=x_data.device, dtype=x_data.dtype)
        J_t = torch.tensor(J, device=x_data.device, dtype=x_data.dtype)
        for _ in range(K_max):
            m_new = torch.mean(torch.tanh(x_data + J_t * m), dim=1, keepdim=True)
            if torch.max(torch.abs(m_new - m)).item() < 1e-8:
                m = m_new
                break
            m = m_new
        y = torch.tanh(x_data + J_t * m)
        s_bar = float(torch.mean(1.0 - y ** 2).item())
    return J * s_bar, s_bar

## Smoke Test

Validates IFT gradient correctness on small inputs (n=64, B=4, K=10):
- IFT and Unrolled outputs must agree to within 1e-4
- No NaN gradients
- If GPU available: memory sanity check that Unrolled > IFT at n=256

In [ ]:
logger.info("=== Smoke Test (n=64, B=4, K=10) ===")
torch.manual_seed(42)
B, n = 4, 64
x_small = torch.randn(B, n, device=DEVICE, requires_grad=True)
J_raw = torch.tensor(2.0, device=DEVICE, requires_grad=True)  # smaller J for stability
J_val = torch.sigmoid(J_raw)

# IFT forward
y_ift = cwa_ift_forward(x_small, J_val, K_max=10, tol=1e-6)
loss_ift = y_ift.sum()
loss_ift.backward()
grad_x_ift = x_small.grad.clone()
grad_J_ift = J_raw.grad.clone()

logger.info(f"IFT output norm: {y_ift.norm().item():.4f}")
logger.info(f"IFT grad_x norm: {grad_x_ift.norm().item():.4f}")
logger.info(f"IFT grad_J: {grad_J_ift.item():.4f}")
assert not torch.isnan(grad_x_ift).any(), "NaN in IFT grad_x"
assert not torch.isnan(grad_J_ift).any(), "NaN in IFT grad_J"

# Unrolled forward
x_small2 = x_small.detach().clone().requires_grad_(True)
J_raw2 = torch.tensor(2.0, device=DEVICE, requires_grad=True)
J_val2 = torch.sigmoid(J_raw2)
y_unrolled = cwa_unrolled_forward(x_small2, J_val2, K_max=10)
loss_unrolled = y_unrolled.sum()
loss_unrolled.backward()
grad_x_unrolled = x_small2.grad.clone()

# Check output agreement
output_diff = (y_ift.detach() - y_unrolled.detach()).abs().max().item()
logger.info(f"IFT vs Unrolled output max diff: {output_diff:.6f} (expect < 1e-4)")
assert output_diff < 1e-4, f"IFT/Unrolled output mismatch: {output_diff}"

# Check gradient agreement (should be close since K=10 is near-converged)
grad_diff = (grad_x_ift - grad_x_unrolled).abs().max().item()
logger.info(f"IFT vs Unrolled grad_x max diff: {grad_diff:.4f}")

# Regime check
J_sigmoid_val = torch.sigmoid(torch.tensor(4.0)).item()
x01 = torch.randn(64, 256, device=DEVICE) * 0.1
x10 = torch.randn(64, 256, device=DEVICE) * 1.0
jsbar_01, sbar_01 = compute_jsbar(x01, J_sigmoid_val)
jsbar_10, sbar_10 = compute_jsbar(x10, J_sigmoid_val)
logger.info(f"x_scale=0.1: J*s̄={jsbar_01:.4f}, s̄={sbar_01:.4f} (expect J*s̄>0.8 near-critical)")
logger.info(f"x_scale=1.0: J*s̄={jsbar_10:.4f}, s̄={sbar_10:.4f} (expect J*s̄<0.5 saturated)")

if HAS_GPU:
    # Quick memory sanity at n=256
    n_q = 256
    B_q = 64
    x_q = torch.randn(B_q, n_q, device=DEVICE) * 0.1

    def quick_ift():
        xi = x_q.clone().requires_grad_(True)
        Ji = torch.tensor(4.0, device=DEVICE, requires_grad=True)
        Jv = torch.sigmoid(Ji)
        return cwa_ift_forward(xi, Jv, K_max=50, tol=1e-6)

    def quick_unrolled():
        xi = x_q.clone().requires_grad_(True)
        Ji = torch.tensor(4.0, device=DEVICE, requires_grad=True)
        Jv = torch.sigmoid(Ji)
        return cwa_unrolled_forward(xi, Jv, K_max=50)

    mem_ift, _ = measure_peak_memory_mb(quick_ift, n_warmup=2, n_measure=3)
    mem_unrolled, _ = measure_peak_memory_mb(quick_unrolled, n_warmup=2, n_measure=3)
    logger.info(f"Sanity n=256: IFT={mem_ift:.1f}MB  Unrolled={mem_unrolled:.1f}MB  ratio={mem_unrolled/mem_ift:.2f}x")
    assert mem_unrolled > mem_ift, "Unrolled must use more memory than IFT"

logger.info("Smoke test PASSED")

## Main Benchmark (GPU Required)

Sweeps over `WIDTHS × X_SCALES`, measuring peak GPU memory for each of the three methods.  
If no GPU is available, this cell is skipped and pre-computed results are used for visualization.

In [ ]:
live_results = []  # populated if GPU available; otherwise we use loaded data

if not HAS_GPU:
    print("No GPU detected — skipping live benchmark. Pre-computed results will be used.")
else:
    logger.info("CWA Memory Benchmark starting")

    for n in WIDTHS:
        for x_scale in X_SCALES:
            logger.info(f"--- n={n}, x_scale={x_scale} ---")
            torch.manual_seed(0)
            x_data = torch.randn(BATCH, n, device=DEVICE) * x_scale

            # Compute regime diagnostics
            jsbar, sbar = compute_jsbar(x_data, J_sigmoid)
            logger.info(f"  J*s̄={jsbar:.4f}, s̄={sbar:.4f}")
            if jsbar >= 1.0:
                logger.warning(f"  J*s̄={jsbar:.4f} >= 1.0 — IFT instability risk. Reducing J_raw to 3.0")
                J_eff = float(torch.sigmoid(torch.tensor(3.0)).item())
                jsbar, sbar = compute_jsbar(x_data, J_eff)
            else:
                J_eff = J_sigmoid

            # ── GELU ──────────────────────────────────────────────────────────
            gelu_model = GELUBaseline(n).to(DEVICE)

            def gelu_fn():
                xi = x_data.clone().requires_grad_(True)
                return gelu_model(xi)

            mem_gelu, std_gelu = measure_peak_memory_mb(gelu_fn, n_warmup=N_WARMUP, n_measure=N_MEASURE)
            logger.info(f"  GELU:     {mem_gelu:.1f} ± {std_gelu:.1f} MB")
            del gelu_model
            gc.collect()
            torch.cuda.empty_cache()

            # ── IFT ───────────────────────────────────────────────────────────
            def ift_fn():
                xi = x_data.clone().requires_grad_(True)
                Ji = torch.tensor(J_RAW_FIXED, device=DEVICE, requires_grad=True)
                Jv = torch.sigmoid(Ji)
                return cwa_ift_forward(xi, Jv, K_max=K_MAX, tol=TOL)

            mem_ift, std_ift = measure_peak_memory_mb(ift_fn, n_warmup=N_WARMUP, n_measure=N_MEASURE)
            logger.info(f"  IFT:      {mem_ift:.1f} ± {std_ift:.1f} MB")
            gc.collect()
            torch.cuda.empty_cache()

            # ── Unrolled K_MAX ─────────────────────────────────────────────────
            def unrolled_fn():
                xi = x_data.clone().requires_grad_(True)
                Ji = torch.tensor(J_RAW_FIXED, device=DEVICE, requires_grad=True)
                Jv = torch.sigmoid(Ji)
                return cwa_unrolled_forward(xi, Jv, K_max=K_MAX)

            mem_unrolled, std_unrolled = measure_peak_memory_mb(
                unrolled_fn, n_warmup=N_WARMUP, n_measure=N_MEASURE
            )
            logger.info(f"  Unrolled: {mem_unrolled:.1f} ± {std_unrolled:.1f} MB")
            gc.collect()
            torch.cuda.empty_cache()

            # Ratios
            ratio_ift_gelu     = mem_ift / mem_gelu if mem_gelu > 0 else float("inf")
            ratio_ift_unrolled = mem_ift / mem_unrolled if mem_unrolled > 0 else float("inf")
            ratio_unrolled_gelu = mem_unrolled / mem_gelu if mem_gelu > 0 else float("inf")

            logger.info(
                f"  IFT/GELU={ratio_ift_gelu:.2f}x  "
                f"IFT/Unrolled={ratio_ift_unrolled:.2f}x  "
                f"Unrolled/GELU={ratio_unrolled_gelu:.2f}x"
            )

            live_results.append({
                "n": n,
                "x_scale": x_scale,
                "J": J_eff,
                "J_raw": J_RAW_FIXED,
                "Jsbar": jsbar,
                "sbar": sbar,
                "peak_MB_gelu": mem_gelu,
                "peak_MB_ift": mem_ift,
                "peak_MB_unrolled": mem_unrolled,
                "std_gelu": std_gelu,
                "std_ift": std_ift,
                "std_unrolled": std_unrolled,
                "ratio_ift_over_gelu": ratio_ift_gelu,
                "ratio_ift_over_unrolled": ratio_ift_unrolled,
                "ratio_unrolled_over_gelu": ratio_unrolled_gelu,
            })

            del x_data
            gc.collect()
            torch.cuda.empty_cache()

    logger.info(f"Live benchmark complete: {len(live_results)} configs measured.")

## Results Visualization

Using **pre-computed results** (full benchmark: n∈{256,1024,4096}, both regimes, K_max=50, N_measure=5) loaded from `mini_demo_data.json`.

If you ran the live benchmark above, `live_results` will be shown alongside for comparison.

In [ ]:
# ── Extract summary table from loaded data ───────────────────────────────────
summary = meta["summary_table"]

# Print text table
print(f"{'n':>6} {'x_sc':>5} {'J*s̄':>6} {'GELU MB':>9} {'IFT MB':>8} {'Unrolled MB':>12} {'IFT/GELU':>9} {'IFT/Unrl':>9}")
print("-" * 72)
for r in summary:
    print(
        f"{r['n']:>6} {r['x_scale']:>5} {r['Jsbar']:>6.3f} "
        f"{r['gelu_MB']:>9.1f} {r['ift_MB']:>8.1f} {r['unrolled_MB']:>12.1f} "
        f"{r['ift_over_gelu']:>9.3f} {r['ift_over_unrolled']:>9.3f}"
    )

print("\nFinding:", meta["finding"])

# ── Bar chart: peak memory vs width for x_scale=0.1 ─────────────────────────
rows_01 = [r for r in summary if r["x_scale"] == 0.1]
widths  = [r["n"] for r in rows_01]
gelu_mb = [r["gelu_MB"] for r in rows_01]
ift_mb  = [r["ift_MB"] for r in rows_01]
unrl_mb = [r["unrolled_MB"] for r in rows_01]

x_pos = np.arange(len(widths))
bar_w = 0.25

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# Left: absolute MB
ax = axes[0]
ax.bar(x_pos - bar_w, gelu_mb, bar_w, label="GELU",     color="#4C72B0")
ax.bar(x_pos,         ift_mb,  bar_w, label="CWA-IFT",  color="#DD8452")
ax.bar(x_pos + bar_w, unrl_mb, bar_w, label="Unrolled K=50", color="#55A868")
ax.set_yscale("log")
ax.yaxis.set_major_formatter(mticker.ScalarFormatter())
ax.set_xticks(x_pos)
ax.set_xticklabels([f"n={w}" for w in widths])
ax.set_xlabel("Layer Width n")
ax.set_ylabel("Peak GPU Memory (MB, log scale)")
ax.set_title("Peak Memory: x_scale=0.1 (near-critical)")
ax.legend()
ax.grid(axis="y", alpha=0.4)

# Right: IFT/Unrolled ratio vs n
ax2 = axes[1]
ratios = [r["ift_over_unrolled"] for r in rows_01]
ax2.plot(widths, ratios, "o-", color="#DD8452", linewidth=2, markersize=8)
ax2.axhline(1.0, linestyle="--", color="gray", alpha=0.6, label="IFT = Unrolled")
ax2.set_xscale("log", base=2)
ax2.set_xticks(widths)
ax2.set_xticklabels([str(w) for w in widths])
ax2.set_xlabel("Layer Width n")
ax2.set_ylabel("IFT / Unrolled Memory Ratio")
ax2.set_title("IFT Memory Savings vs Unrolled K=50")
ax2.legend()
ax2.grid(alpha=0.4)
for xi, yi in zip(widths, ratios):
    ax2.annotate(f"{100*(1-yi):.0f}% saved", (xi, yi),
                 textcoords="offset points", xytext=(6, -12), fontsize=9)

plt.tight_layout()
plt.savefig("memory_benchmark.png", dpi=100, bbox_inches="tight")
plt.show()
print("Plot saved to memory_benchmark.png")

# Live results comparison if available
if live_results:
    print("\n── Live Benchmark Results ──")
    print(f"{'n':>6} {'x_sc':>5} {'GELU MB':>9} {'IFT MB':>8} {'Unrolled MB':>12} {'IFT/Unrl':>9}")
    print("-" * 55)
    for r in live_results:
        print(
            f"{r['n']:>6} {r['x_scale']:>5} {r['peak_MB_gelu']:>9.1f} "
            f"{r['peak_MB_ift']:>8.1f} {r['peak_MB_unrolled']:>12.1f} "
            f"{r['ratio_ift_over_unrolled']:>9.3f}"
        )